In [13]:
def quantize(floatList):

    q_min, q_max = -128, 127

    x_min = min(floatList)
    x_max = max(floatList)

    scale = (x_max - x_min) / (q_max - q_min)

    zero_point = round(q_min - x_min / scale)

    quantizedList = []

    for xnum in floatList:

        qnum = round(xnum / scale + zero_point)

        if qnum < q_min:
            qnum = q_min
        elif qnum > q_max:
            qnum = q_max

        quantizedList.append(qnum)

    return quantizedList, scale, zero_point



size = int(input("Enter the size of the floating array: "))

floatList = []

for i in range(size):
    num = float(input(f"Enter number {i + 1}: "))
    floatList.append(num)

# Quantization
quantizedList, scale, zero_point = quantize(floatList)

print("Your Float List:", floatList)
print("Scale:", scale)
print("Zero Point:", zero_point)
print("Your Quantized Int8 List:", quantizedList)

Enter the size of the floating array: 5
Enter number 1: 9
Enter number 2: 9.5
Enter number 3: 1
Enter number 4: 1.4
Enter number 5: 2.5
Your Float List: [9.0, 9.5, 1.0, 1.4, 2.5]
Scale: 0.03333333333333333
Zero Point: -158
Your Quantized Int8 List: [112, 127, -128, -116, -83]


The float values were converted into the int8 range from -128 to 127 and the quantized values are not exactly the same as the original values because of the loss of precision.So we can say that this reduces the memory needed to store the values.


In [11]:
dtype_sizes = {"fp32": 4, "fp16": 2, "int8": 1, "int4": 0.5}

def calculate_model_size(no_of_params, dtype: str):
  reqBytes = dtype_sizes[dtype]
  modelSize = no_of_params * reqBytes
  return modelSize

#user input
# params = int(input("Enter the number of parameters: "))
# dtype = input("Enter the data type of the parameters(fp32, fp16, int8, int4): ")

# modelSize = calculate_model_size(params,dtype)

# modelSize_GB = modelSize/(1024**3)

# print("Model size: ",modelSize_GB,"GB")

model1 = calculate_model_size(125000000,"fp32")
model2 = calculate_model_size(1000000000,"fp16")
model3 = calculate_model_size(8000000000,"int8")
model4 = calculate_model_size(14000000000,"int4")

model1GB = model1/(1024**3)
model2GB = model2/(1024**3)
model3GB = model3/(1024**3)
model4GB = model4/(1024**3)

print(f"Model 1 size: {model1GB:.2f} GB")
print(f"Model 2 size: {model2GB:.2f} GB")
print(f"Model 3 size: {model3GB:.2f} GB")
print(f"Model 4 size: {model4GB:.2f} GB")

Model 1 size: 0.47 GB
Model 2 size: 1.86 GB
Model 3 size: 7.45 GB
Model 4 size: 6.52 GB


Larger models require more memory to store their parameters and FP16 uses half the memory of FP32 while INT8 uses one-fourth of the memory.
So, converting FP32 to INT8 reduces the model size by about 75%.


### 1. Where would you split the model (within a node vs across nodes), and why?

As per my knowledge I would always try to keep all the gpus in one node itself and only and only if the model is still too large then i would consider splitting them across nodes, So like I would first split the model across the 8 GPUs within the same node. Since these GPUs are connected using NVLink, they can communicate very quickly. If the model is still too large for the 8 GPUs, then I would split it across multiple nodes using InfiniBand.

### 2. What role does NVLink play vs InfiniBand in this setup?

So basically NVLink is used for fast communication between GPUs within the same node and InfiniBand is used for communication between different nodes. In simple terms, NVLink handles GPU to GPU communication inside a node, while InfiniBand handles node to node communication.

### 3. What would go wrong if you swapped their roles?

If InfiniBand was used for communication between GPUs inside the same node, it would add unnecessary network overhead and increase communication latency. The GPUs are already connected through NVLink, so using InfiniBand would make their communication less efficient.
Similarly, NVLink is not meant to connect separate nodes. If we tried to use it for communication between different nodes, it would not work as a replacement for the network connection provided by InfiniBand. Since model parallelism can require GPUs to exchange data frequently, slower communication would cause GPUs to spend more time waiting for data, which would reduce overall inference performance.

### 4. Write your answer as a short design note (half a page), no code needed.

For a model that is too large to fit on a single GPU, I would first distribute it across the 8 GPUs within the same node. These GPUs are connected through NVLink, which provides a fast connection between them. Keeping the model within one node as much as possible is useful because the GPUs may need to exchange intermediate data frequently during inference.
If the model is still too large for the 8 GPUs, I would then distribute it across multiple nodes. The nodes are connected using InfiniBand, so it can handle the communication between GPUs that are located on different nodes.
The main idea is to use each connection for the communication it is designed for. NVLink should handle communication between GPUs within the same node, while InfiniBand should handle communication between separate nodes. If these roles were swapped, communication would become less efficient. Using InfiniBand between GPUs in the same node would introduce unnecessary network overhead and higher latency, while NVLink cannot replace the network connection needed between separate nodes. This could make communication a bottleneck and reduce the overall inference performance.


In [12]:
def compute_or_memory(flops_per_token,compute_rate,bytes_per_token, bandwidth):
  compute_time = flops_per_token / compute_rate
  memory_time = bytes_per_token / bandwidth

  if compute_time > memory_time:
      bound = "Compute-bound"
  elif memory_time > compute_time:
      bound = "Memory-bound"
  else:
      bound = "Balanced"

  return bound, compute_time, memory_time

eg1 = compute_or_memory(200e9,20e12,10e9,1e12)
print("Example 1:")
print("Type:", eg1[0])
print(f"Compute time: {eg1[1] * 1000:.2f} ms")
print(f"Memory time:  {eg1[2] * 1000:.2f} ms")

eg2 = compute_or_memory(500e9, 20e12, 10e9, 1e12)
print("Example 2:")
print("Type:", eg2[0])
print(f"Compute time: {eg2[1] * 1000:.2f} ms")
print(f"Memory time:  {eg2[2] * 1000:.2f} ms")

eg3 = compute_or_memory(100e9, 20e12, 20e9, 1e12)
print("Example 3:")
print("Type:", eg3[0])
print(f"Compute time: {eg3[1] * 1000:.2f} ms")
print(f"Memory time:  {eg3[2] * 1000:.2f} ms")



Example 1:
Type: Balanced
Compute time: 10.00 ms
Memory time:  10.00 ms
Example 2:
Type: Compute-bound
Compute time: 25.00 ms
Memory time:  10.00 ms
Example 3:
Type: Memory-bound
Compute time: 5.00 ms
Memory time:  20.00 ms


The result depends on whether computation time or memory access time is higher.
A compute-bound model benefits more from reducing FLOPs or improving compute efficiency.
A memory-bound model benefits more from reducing memory traffic, such as through quantization.
This helps identify what type of optimization would be more useful for a particular setup.